Da das mit der msa nicht geklappt hat, war der Plan jetzt über eine API nur die Sequnz der Genfamilien zu downloaden und damit dann erneut eine msa zu machen. Vermustlich handelt es sich bei den anderen Sequenzen um die ganze hc, weshalb die msa nicht erfolgreich war. 

In [5]:
from Bio import SeqIO
import pandas as pd

# Pfade
mapping_path = "../../documentation/clean_entity_mapping.csv"
fasta_path = "../../documentation/pdb_sequences.fasta"
output_path = "heavy_chains_with_families.fasta"

# Mapping-Datei laden
mapping_df = pd.read_csv(mapping_path, dtype=str)
mapping_df = mapping_df[mapping_df["H_entity_id"].notna()]
mapping_df["pdb"] = mapping_df["pdb"].str.lower().str.strip()
mapping_df["H_entity_id"] = mapping_df["H_entity_id"].str.strip()

# Mapping: (pdb_id, H_entity_id) → (heavy_subclass, light_subclass)
entity_to_families = {
    (row["pdb"], row["H_entity_id"]): (row["heavy_subclass"], row["light_subclass"])
    for _, row in mapping_df.iterrows()
    if row["H_entity_id"].isdigit()
}

print(f"Mapping-Paare mit Subklassen: {len(entity_to_families)}")

count_total = 0
count_written = 0

with open(output_path, "w") as out_fasta:
    for record in SeqIO.parse(fasta_path, "fasta"):
        count_total += 1

        header = record.id
        sequence = str(record.seq)

        # Header aufteilen: >5GGT_1|A|Homo sapiens
        try:
            pdb_entity_part, rest = header.split("|", 1)
            pdb_id, entity_id = pdb_entity_part.split("_")
            pdb_id = pdb_id.lower().strip()
            entity_id = entity_id.strip()
        except ValueError:
            continue  # falls der Header nicht passt

        key = (pdb_id, entity_id)
        if key in entity_to_families:
            h_sub, l_sub = entity_to_families[key]
            # Neuen Header schreiben
            new_header = f"{header}|{h_sub}|{l_sub}"
            out_fasta.write(f">{new_header}\n{sequence}\n")
            count_written += 1

print(f"Insgesamt {count_total} Sequenzen geprüft.")
print(f"{count_written} Heavy Chain Sequenzen mit Subklassen in '{output_path}' gespeichert.")

Mapping-Paare mit Subklassen: 1442
Insgesamt 4317 Sequenzen geprüft.
1442 Heavy Chain Sequenzen mit Subklassen in 'heavy_chains_with_families.fasta' gespeichert.


folgender Code gibt mit Hilfe von anarci die variable domain der Sequez an.

In [1]:
from anarci import anarci

# Eingabesequenz
seq_name = "5wi9_entity3"
seq = "QVQLVESGGGVVQPGRSLRLSCAASGFTFSNYGIHWVRQAPGKGLEWVAVIWYDGSIKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDRAAAGLHYYYGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPK"



# ANARCI-Lauf mit IMGT-Schema
result = anarci([(seq_name, seq)], scheme='imgt')

# Ergebnisse entpacken
numbering = result[0][0][0]  # Liste von (position, aa)
domain_info = result[2][0]   # Metadaten: [header, domain1, domain2, ...]

# Gesuchte Domain
gesuchte_domain_id = "human_H"

# Domain finden
for domain in domain_info[1:]:  # Erste Zeile ist Header
    domain_id = domain[domain_info[0].index('id')]
    if domain_id == gesuchte_domain_id:
        start = int(domain[domain_info[0].index('query_start')])
        end = int(domain[domain_info[0].index('query_end')])
        print(f"✅ Gefunden: {domain_id} von Position {start} bis {end}")
        print(f"Variable Region:\n{seq[start:end+1]}")
        break
else:
    print("❌ Keine passende Domain gefunden.")

✅ Gefunden: human_H von Position 0 bis 124
Variable Region:
QVQLVESGGGVVQPGRSLRLSCAASGFTFSNYGIHWVRQAPGKGLEWVAVIWYDGSIKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDRAAAGLHYYYGMDVWGQGTTVTVSSA


hier habe ich nach dem Test das ganze auf alle Sequenzen der heavy chains aus der Datei heavy_chains_with_families.csv angewendet. 

In [7]:
import csv
from anarci import anarci

# === 1. FASTA-Datei lesen und in CSV umwandeln ===
fasta_path = "heavy_chains_with_families.fasta"
csv_path = "heavy_chains_with_families.csv"

entries = []
with open(fasta_path, "r") as fasta_file:
    name = None
    seq_lines = []
    for line in fasta_file:
        line = line.strip()
        if line.startswith(">"):
            if name and seq_lines:
                sequence = "".join(seq_lines)
                entries.append((name, sequence))
            name = line[1:].strip()  # Header ohne ">"
            seq_lines = []
        else:
            seq_lines.append(line)
    # Letzten Eintrag speichern
    if name and seq_lines:
        entries.append((name, "".join(seq_lines)))

# In CSV schreiben
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["seq_name", "sequence"])
    writer.writerows(entries)

print(f"✅ {len(entries)} Sequenzen in {csv_path} gespeichert.")

# === 2. CSV lesen und ANARCI anwenden ===
results = []

with open(csv_path, newline="") as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        seq_name = row["seq_name"].strip('"')  # Quotes entfernen, falls vorhanden
        sequence = row["sequence"]

        try:
            anarci_result = anarci([(seq_name, sequence)], scheme='imgt')
            numbering = anarci_result[0][0][0]
            domains = anarci_result[2][0]

            if len(domains) > 1:
                for domain in domains[1:]:
                    domain_id = domain[domains[0].index("id")]
                    start = int(domain[domains[0].index("query_start")])
                    end = int(domain[domains[0].index("query_end")])
                    variable_region = sequence[start:end+1]
                    results.append([seq_name, domain_id, start, end, variable_region])
            else:
                results.append([seq_name, "not_found", "", "", "domain_not_found"])

        except Exception as e:
            results.append([seq_name, "error", "", "", f"{e}"])

# === 3. Ergebnisse speichern ===
with open("anarci_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["seq_name", "domain_id", "start", "end", "variable_region"])
    writer.writerows(results)

print(f"✅ Ergebnisse in anarci_results.csv gespeichert.")

✅ 1442 Sequenzen in heavy_chains_with_families.csv gespeichert.
✅ Ergebnisse in anarci_results.csv gespeichert.
